##### Word2Vec (the big picture)
- Unlike BoW/TF-IDF, which represent words as sparse counts, Word2Vec learns a dense, low-dimensional vector for each word (e.g. 50–300 numbers) by training a tiny neural network to predict words from their context. 
- Words used in similar contexts end up with similar vectors — so the vectors capture meaning, not just frequency.

In [1]:
import pandas as pd

data = pd.read_csv("../data/fd_dataset_messy.csv")
print(data.head(5))

                         email                 time        subject  \
0  mukesh.bhatt@rediffmail.com  2025-02-17 20:29:47  Payment issue   
1      sanjay.jain@hotmail.com  2024-06-07 16:22:21           Help   
2         ashok.nair@gmail.com  2024-04-05 08:16:39     Loan query   
3    manoj.gupta51@hotmail.com  2025-01-04 23:24:05          Query   
4     vinod.chopra87@gmail.com  2025-03-03 10:12:43    Loan status   

                                             content              label  
0  Branch gaya tha, unhone email karne bola. Two ...             Non-FD  
1  Dear customer care, My insurance policy is rea...             Non-FD  
2  Namaskar, Rs 6 lakh kat gaya bina bataye insur...             Non-FD  
3  Dear customer care, 1. Suna hai Bajaj Finance ...  Multiple Category  
4  Dear customer care, Where is my money? The rev...             Non-FD  


In [2]:
import re
from gensim.models import Word2Vec

def tokenize(s):
    return re.findall(r'\b[a-zA-Z]+\b', s.lower())

sentences = [tokenize(t) for t in data['content'].astype(str)]

print("First Email Sample:", data['content'][0])
print("\n")
print("Word2Vec ENCODING")
print(sentences[0])

First Email Sample: Branch gaya tha, unhone email karne bola. Two wheeler loan ki EMI is mahine zyada kati hai. Pehle se Rs 800 jyada. Kyu? Statement bhejo. Bahut pareshan ho gaya hoon. Mukesh Bhatt


Word2Vec ENCODING
['branch', 'gaya', 'tha', 'unhone', 'email', 'karne', 'bola', 'two', 'wheeler', 'loan', 'ki', 'emi', 'is', 'mahine', 'zyada', 'kati', 'hai', 'pehle', 'se', 'rs', 'jyada', 'kyu', 'statement', 'bhejo', 'bahut', 'pareshan', 'ho', 'gaya', 'hoon', 'mukesh', 'bhatt']


Note: Word2Vec needs a list of tokenized sentences (list of lists of words), not raw strings — that's the key input difference from CountVectorizer/TfidfVectorizer.

##### CBOW (Continuous Bag of Words)
- CBOW predicts the middle word from its surrounding context words. 
- E.g. given ["wheeler", "___", "ki", "emi"], it tries to predict "loan". It's faster to train and tends to work fine for frequent words.

In [3]:
cbow_model = Word2Vec(sentences, vector_size=50, window=5, min_count=2, sg=0, epochs=50)

##### Hyperparameter Explaination

Parameter 1 — vector_size=50
- Each word gets represented as a list of 50 numbers (a 50-dimensional vector). This is just the "size" of that fingerprint — bigger vectors can capture more nuance, but need more training data to fill them meaningfully. 
- Your vocabulary only has 706 words after filtering, from 2500 short emails — that's a small corpus, so 50 is already a modest, reasonable choice. Going up to something like 300 (common for huge corpora) would just give the model far more numbers to tune than your data can actually justify.

Parameter 2 — window=5
- This sets how many words on each side count as "context" for a target word. I checked it directly on row 0, predicting "emi":
    - before: ['bola', 'two', 'wheeler', 'loan', 'ki']
    - after:  ['is', 'mahine', 'zyada', 'kati', 'hai']
- With window=5, CBOW uses all 10 of those words to try to predict "emi". If you set window=2, it would only use ['loan', 'ki'] and ['is', 'mahine'] — a much tighter, more local context.

Parameter 3 — min_count=2
- Only words appearing at least twice across the whole corpus get a vector at all, everything rarer gets ignored completely (no vector, can't appear in most_similar, etc). 
- I counted it: your corpus has 902 unique words total, and 196 of them appear exactly once — mostly typos like 'payent', 'sttement', 'screensot', 'uresh'. Those get dropped, leaving 706 words that actually get trained — which matches the vocabulary size you saw earlier (len(cbow_model.wv.index_to_key) = 706).

Parameter 4 — sg=0
- This is the architecture switch: 
    - sg=0 → CBOW (predict the middle word from its context — what we've been doing). 
    - sg=1 → Skip-gram (predict the context from the middle word — the one we compared it against earlier).

Parameter 5 — epochs=50
- This is how many full passes the model makes over your entire dataset before stopping. 
- Each pass slides the context window over every word in every one of your 2500 emails and nudges the vectors slightly closer to "correct." 50 full passes means it sees every sentence 50 times. 
- On a small, repetitive dataset like yours, more epochs means the model leans even harder into whatever patterns repeat most — which is part of why "loan" ended up so tightly bound to words from its repeated template sentences ("outstanding", "extend") rather than broader synonyms.

Quick rule of thumb if you want to tune this later:
- lower min_count keeps more rare/typo words (noisier vectors)
- higher window captures broader topic context but blurs local word order
- more epochs sharpens patterns but risks overfitting on a small/templated corpus like this one.

In [4]:
# sg=0 -> CBOW
print(cbow_model.wv.most_similar('loan', topn=5))

[('reasons', 0.4905463457107544), ('outstanding', 0.41797205805778503), ('extend', 0.4117428958415985), ('now', 0.4002639651298523), ('foreclosure', 0.38438719511032104)]


##### Hyperprameter Explaination

cbow_model.wv.most_similar('loan', topn=5)

-  .wv: 
    - This stands for "word vectors." 
    - It's the part of the trained model that actually stores every word's 50-number vector and knows how to compare them. 
-  .most_similar('loan', ...): 
    - This takes "loan"'s 50-dimensional vector, compares it against the vector of every other word in the vocabulary using cosine similarity, and sorts the results from most similar to least. 
    - I checked — your vocabulary has 706 words, so this is comparing "loan" against the other 705 candidates.
- topn=5: 
    - This just caps how many results come back. topn=5 → top 5 matches. 
    - If you drop the argument entirely, gensim defaults to topn=10:

##### Output Explaination
Note 1 — What CBOW is actually trying to learn
- For every word in every email, CBOW looks at the words around it (the "context window") and tries to predict the missing middle word. 
- Example from your data: given context ["wheeler", "ki", "emi"], it tries to guess the missing word is "loan". 
- It does this for every word, in every one of your 2500 emails, repeatedly (you set epochs=50, so it goes over the whole dataset 50 times).

Note 2 — Where the "vector" actually comes from
- Internally, this prediction task is done by a small neural network with one hidden layer of size 50 (vector_size=50). 
- To get good at predicting target words from context, the network is forced to assign each word a 50-number dimension — they're just whatever pattern the network settled on while trying to get good at the fill-in-the-blank task. 
- The word vector is a side-effect of training, not the goal itself.

Note 3 — What most_similar('loan') actually computes
- Once training is done, every word has its own 50-number vector. 
- most_similar just measures the angle (cosine similarity) between "loan"'s vector and every other word's vector, and returns the closest ones. 

So your output:
- [('reasons', 0.458), ('outstanding', 0.418), ('now', 0.376), ('extend', 0.372), ('interst', 0.368)]
- means: after 50 rounds of fill-in-the-blank training, the vector the network landed on for "loan" happens to point in roughly the same direction as the vectors for "reasons", "outstanding", "now", "extend", and "interst" (misspelling of "interest").

Note 4 — Why specifically these words (verified against your actual data)
- I checked your raw emails, and the reason isn't subtle — it's literal repetition:
- So "loan", "outstanding", "extend", and "reasons" keep showing up inside the same handful of repeated template sentences, just a few words apart. 
- Every time CBOW trains on one of those windows, it nudges these words' vectors toward each other a little. 
- With the same templates repeating dozens of times across the 2500 rows, those nudges add up — which is exactly why they end up as each other's nearest neighbors.

##### Skip-gram
- Skip-gram does the opposite — it predicts the context words from the middle word. 
- Given "loan", it tries to predict "wheeler", "ki", "emi", etc. 
- It's slower to train but generally does better on rare words and smaller datasets — which matters here, since your dataset is small (2500 short emails) and mixes Hindi+English.

In [5]:
sg_model = Word2Vec(sentences, vector_size=50, window=5, min_count=2, sg=1, epochs=50, workers=1)
# sg=1 -> Skip-gram
print(sg_model.wv.most_similar('loan', topn=5))

[('personal', 0.5468371510505676), ('noc', 0.5017400979995728), ('aganst', 0.49028924107551575), ('against', 0.45805981755256653), ('home', 0.4530121386051178)]


For same word 'loan', top 5 similar words output are different?

CBOW Output - [('reasons', 0.47020256519317627), ('extend', 0.4408171474933624), ('interst', 0.4379308521747589), ('now', 0.41549474000930786), ('outstanding', 0.4053826332092285)]

Skip Gram Output - [('personal', 0.5468371510505676), ('noc', 0.5017400979995728), ('aganst', 0.49028924107551575), ('against', 0.45805981755256653), ('home', 0.4530121386051178)]

Why?
- CBOW and Skip-gram learn in opposite directions, so they end up building different vectors for the same word.
- CBOW: looks at context words → guesses the middle word ("loan").
- Skip-gram: looks at "loan" → guesses each context word around it, one at a time.
- Because the training task is different, the math that shapes each word's vector is different — so "closest words to loan" comes out different too.
- Skip-gram trains on each context word individually, so it usually learns rarer/specific words a bit better than CBOW, which is why its neighbors here ("personal", "noc", "against") look slightly more on-topic than CBOW's.

##### Average Word2Vec (turning a whole email into one vector)
- Word2Vec only gives you a vector per word. 
- To classify a whole email as FD/Non-FD, you need one fixed-length vector per email. 
- The simplest approach: average the vectors of every word in that email.

In [6]:
import numpy as np

def avg_word2vec(tokens, model):
    vectors = [model.wv[w] for w in tokens if w in model.wv]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

vec0 = avg_word2vec(sentences[0], sg_model)   # loan/EMI complaint
print("First Email Sample:", data['content'][0])
print("\n")
print("Tokens:", sentences[0])
print("\n")
print("Average Word2Vec:", vec0)

First Email Sample: Branch gaya tha, unhone email karne bola. Two wheeler loan ki EMI is mahine zyada kati hai. Pehle se Rs 800 jyada. Kyu? Statement bhejo. Bahut pareshan ho gaya hoon. Mukesh Bhatt


Tokens: ['branch', 'gaya', 'tha', 'unhone', 'email', 'karne', 'bola', 'two', 'wheeler', 'loan', 'ki', 'emi', 'is', 'mahine', 'zyada', 'kati', 'hai', 'pehle', 'se', 'rs', 'jyada', 'kyu', 'statement', 'bhejo', 'bahut', 'pareshan', 'ho', 'gaya', 'hoon', 'mukesh', 'bhatt']


Average Word2Vec: [-0.02465437  0.07309119  0.52954775  0.08364662  0.07789189 -0.03345431
  0.44228524  0.22869259  0.1914498  -0.35631353 -0.17744276 -0.14857207
  0.01524902  0.01551501 -0.4105518  -0.20513667  0.4540065   0.23909876
 -0.58365184 -0.2582947   0.04997833  0.20938042  0.20038116 -0.35341993
  0.23323831  0.17585458 -0.59620017 -0.43863738 -0.49131516  0.34247756
  0.01844942  0.24993277 -0.39760646 -0.41020244  0.24382152  0.6236722
  0.36448675 -0.10436411 -0.00253127 -0.48519504  0.3211444  -0.11339501

To check whether this actually captures meaning, compare FD vs Non FD using cosine similarity:

In [7]:
from numpy.linalg import norm
cosine_sim = lambda a, b: np.dot(a, b) / (norm(a) * norm(b))

# FD Sample 
vec13 = avg_word2vec(sentences[13], sg_model) 
vec17 = avg_word2vec(sentences[17], sg_model) 
print("FD-email vs another-FD-email:", cosine_sim(vec13, vec17))

# Non FD Email Sample
vec1  = avg_word2vec(sentences[1], sg_model)   # Two Wheeler Loan 

print("FD-email vs Loan-email:", cosine_sim(vec17, vec1))

FD-email vs another-FD-email: 0.7598507
FD-email vs Loan-email: 0.669861


This is the inference that matters: 
- FD-email vs another-FD-email: 0.75 but FD-email vs Loan-email: 0.66
- Even though no vectorizer was ever told what "loan" or "insurance" means. 
- That's the whole value proposition of Word2Vec over BoW/TF-IDF: it captures semantic closeness, not just shared vocabulary, and Average Word2Vec is the simplest way to carry that into a per-document feature for your FD/Non-FD classifier.

##### Why Cosine Similarlity?

- "cos" = cosine, the same cosine from trigonometry — it measures the angle between two vectors, not their length.

Why angle, not distance/length?
- A word vector's length doesn't mean anything special — only its direction matters (which other words it points toward). 
- Cosine similarity ignores length completely and just checks: "are these two vectors pointing the same way?"

The scale:
- 1 → same direction (most similar)
- 0 → unrelated (90° apart)
- -1 → opposite direction

So 0.66 for "FD" vs "loan" just means the angle between their vectors is about 66° — fairly close, but not pointing in exactly the same direction.

That's why it's used over plain distance: two vectors can be far apart in raw distance but still point the same way (same meaning) — cosine catches that; distance alone wouldn't.

##### Why not Sine or Tan only Cosine?

Cosine's behavior matches what we want:
- 0° (same direction) → cos = 1 (max similarity) ✓
- 90° (unrelated) → cos = 0 ✓
- 180° (opposite) → cos = -1 ✓
This maps perfectly onto "similar / unrelated / opposite."

Sine would be backwards:
- sin(0°) = 0 → but same direction should mean MOST similar, not 0!
- sin(90°) = 1 → but unrelated should NOT be the highest score!
So sine gives the exact opposite meaning of what we want. Useless here.

Tangent breaks completely:
- tan(90°) = undefined / infinity

Since "unrelated" (90°) is a very common case, tangent would blow up to infinity right at the most normal situation. Also unusable.